In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

rng = np.random.default_rng(20261029)
n = 500


In [2]:
# 500 production-shift observations on weekdays, covering 2025 and part of 2026
dates = pd.bdate_range("2025-01-02", periods=n)
products = np.array(["Geladeira", "Lavadora", "Micro-ondas", "Ar-condicionado"])
product_probs = [0.28, 0.27, 0.20, 0.25]
product = rng.choice(products, size=n, p=product_probs)
shift = rng.choice(["Manhã", "Tarde", "Noite"], size=n, p=[0.38, 0.37, 0.25])
line = []
for p in product:
    if p == "Geladeira": line.append(rng.choice(["Linha A", "Linha B"], p=[0.65,0.35]))
    elif p == "Lavadora": line.append(rng.choice(["Linha B", "Linha C"], p=[0.55,0.45]))
    elif p == "Micro-ondas": line.append(rng.choice(["Linha C", "Linha D"], p=[0.25,0.75]))
    else: line.append(rng.choice(["Linha A", "Linha D"], p=[0.45,0.55]))
line = np.array(line)


In [3]:
# Variables with plausible relationships and some seeded unusual events
planned_hours = rng.choice([7.5, 8.0, 8.5], size=n, p=[0.15,0.70,0.15])
machine_age = rng.integers(1, 13, size=n)
maintenance = rng.choice(["Preventiva", "Corretiva", "Nenhuma"], size=n, p=[0.20,0.08,0.72])
downtime = rng.gamma(shape=1.6, scale=0.32, size=n)
downtime += np.where(maintenance == "Corretiva", rng.uniform(0.5, 1.8, n), 0)
downtime += np.where(machine_age >= 9, rng.uniform(0.1, 0.55, n), 0)
downtime = np.clip(downtime, 0, 4.5)

In [4]:
# Some operational incidents/outliers
incident_idx = rng.choice(n, size=12, replace=False)
downtime[incident_idx] += rng.uniform(1.2, 2.7, size=len(incident_idx))
downtime = np.clip(downtime, 0, 6.5)

ambient_temp = rng.normal(25, 4.5, n).clip(13, 39)
operator_experience = rng.integers(1, 16, n)


In [5]:
# Production rates by product
rate_map = {"Geladeira": 18, "Lavadora": 23, "Micro-ondas": 42, "Ar-condicionado": 25}
base_rate = np.array([rate_map[p] for p in product])
shift_effect = np.select([shift=="Manhã", shift=="Tarde", shift=="Noite"], [1.02,1.0,0.94])
production_hours = np.maximum(planned_hours - downtime, 1.0)
production = np.round(base_rate * shift_effect * production_hours * rng.normal(1,0.075,n)).astype(int)
production = np.maximum(production, 1)


In [6]:
# Defect probability affected by downtime, temperature and maintenance; product-specific baseline
base_defect = np.array([0.035 if p=="Geladeira" else 0.03 if p=="Lavadora" else 0.025 if p=="Micro-ondas" else 0.04 for p in product])
p_def = np.clip(base_defect + 0.007*downtime + 0.0015*np.maximum(ambient_temp-27,0) - 0.0007*(operator_experience-5), 0.008, 0.16)
defects = rng.binomial(production, p_def)


In [7]:
# Seed a few quality incidents
quality_idx = rng.choice(np.setdiff1d(np.arange(n), incident_idx), size=10, replace=False)
defects[quality_idx] += rng.integers(3, 10, size=len(quality_idx))
defects = np.minimum(defects, production)
energy_per_unit = np.array([1.8 if p=="Geladeira" else 1.3 if p=="Lavadora" else 0.45 if p=="Micro-ondas" else 1.6 for p in product])
energy = production * energy_per_unit + downtime*18 + rng.normal(0, 15, n)
energy = np.maximum(energy, 20).round(1)
waste_kg = (defects * np.array([0.85 if p=="Geladeira" else 0.65 if p=="Lavadora" else 0.18 if p=="Micro-ondas" else 0.55 for p in product]) + rng.gamma(1.2,0.7,n)).round(1)
customer_returns = rng.poisson(np.maximum(defects*0.06,0.05))
orders = np.maximum(0, (production * rng.normal(1.03,0.12,n))).round().astype(int)
on_time_pct = np.clip(98 - downtime*3.2 - defects/production*35 + rng.normal(0,2,n), 65, 100).round(1)

df = pd.DataFrame({
    "id_registro": [f"ET{i:04d}" for i in range(1,n+1)],
    "data": dates.strftime("%Y-%m-%d"),
    "turno": shift,
    "linha_producao": line,
    "produto": product,
    "horas_planejadas": planned_hours,
    "horas_parada_maquina": downtime.round(2),
    "idade_maquina_anos": machine_age,
    "tipo_manutencao": maintenance,
    "temperatura_ambiente_c": ambient_temp.round(1),
    "experiencia_operador_anos": operator_experience,
    "unidades_produzidas": production,
    "unidades_defeituosas": defects,
    "consumo_energia_kwh": energy,
    "residuo_kg": waste_kg,
    "devolucoes_clientes": customer_returns,
    "pedidos_recebidos": orders,
    "entregas_no_prazo_pct": on_time_pct
})


In [9]:
# Add a small number of missing values for data-cleaning practice (not in key fields)
for col, count in [("temperatura_ambiente_c", 5), ("experiencia_operador_anos", 4), ("consumo_energia_kwh", 3)]:
    ix = rng.choice(n, size=count, replace=False)
    df.loc[ix, col] = np.nan

out = Path("./eletrotech_base_500_registros.csv")
df.to_csv(out, index=False, encoding="utf-8-sig")
out.exists(), df.shape, df.head(3).to_dict(orient="records")


(True,
 (500, 18),
 [{'id_registro': 'ET0001',
   'data': '2025-01-02',
   'turno': 'Manhã',
   'linha_producao': 'Linha C',
   'produto': 'Lavadora',
   'horas_planejadas': 8.0,
   'horas_parada_maquina': 1.61,
   'idade_maquina_anos': 12,
   'tipo_manutencao': 'Nenhuma',
   'temperatura_ambiente_c': 31.2,
   'experiencia_operador_anos': 8.0,
   'unidades_produzidas': 147,
   'unidades_defeituosas': 6,
   'consumo_energia_kwh': 196.4,
   'residuo_kg': 5.1,
   'devolucoes_clientes': 0,
   'pedidos_recebidos': 157,
   'entregas_no_prazo_pct': 92.8},
  {'id_registro': 'ET0002',
   'data': '2025-01-03',
   'turno': 'Manhã',
   'linha_producao': 'Linha D',
   'produto': 'Ar-condicionado',
   'horas_planejadas': 8.0,
   'horas_parada_maquina': 1.2,
   'idade_maquina_anos': 10,
   'tipo_manutencao': 'Preventiva',
   'temperatura_ambiente_c': 23.7,
   'experiencia_operador_anos': 6.0,
   'unidades_produzidas': 181,
   'unidades_defeituosas': 12,
   'consumo_energia_kwh': 307.2,
   'residuo_kg